In [1]:
import torch
import torch.nn as nn

In [2]:
d_in = 3
d_out = 2

dropout = 0
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.33],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55],
    ]
)

batch = torch.stack([inputs, inputs], dim=0)
context_length = batch.shape[1]

batch.size()

torch.Size([2, 6, 3])

In [3]:
class CausalSelfAttn(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        # For batch
        num_tokens = x.shape[1]
        attn_scores  = queries @ keys.transpose(1,2)

        # For single batch i.e. inputs
        # num_length = 6
        # attn_scores  = queries @ keys.T

        # .bool(): 1 → True, 0 → False; slice to sequence length; replace True positions with -inf.
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec  = attn_weights @ values

        return context_vec



In [4]:
torch.manual_seed(789)
c_attn = CausalSelfAttn(d_in, d_out, context_length, dropout)
c_attn(batch)

tensor([[[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]],

        [[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]]], grad_fn=<UnsafeViewBackward0>)

In [5]:
class MultiHeadAttn(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads = 2, qkv_bias = False):
        super().__init__()

        self.heads = nn.ModuleList([CausalSelfAttn(d_in, d_out, context_length, dropout) for _ in range(num_heads)])

    def forward(self, x):

        return torch.cat([head(x) for head in self.heads], dim = -1)

In [6]:
torch.manual_seed(123)
m_attn = MultiHeadAttn(d_in, d_out, context_length, dropout, num_heads=4)
m_attn(batch)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063,  0.4566,  0.2729, -0.5684,
           0.5063],
         [-0.5874,  0.0058,  0.5891,  0.3257,  0.5792,  0.3011, -0.5388,
           0.6447],
         [-0.6300, -0.0632,  0.6202,  0.3860,  0.6249,  0.3102, -0.5242,
           0.6954],
         [-0.5675, -0.0843,  0.5478,  0.3589,  0.5691,  0.2785, -0.4578,
           0.6471],
         [-0.5526, -0.0981,  0.5321,  0.3428,  0.5543,  0.2520, -0.4006,
           0.5921],
         [-0.5299, -0.1081,  0.5077,  0.3493,  0.5337,  0.2499, -0.3997,
           0.5971]],

        [[-0.4519,  0.2216,  0.4772,  0.1063,  0.4566,  0.2729, -0.5684,
           0.5063],
         [-0.5874,  0.0058,  0.5891,  0.3257,  0.5792,  0.3011, -0.5388,
           0.6447],
         [-0.6300, -0.0632,  0.6202,  0.3860,  0.6249,  0.3102, -0.5242,
           0.6954],
         [-0.5675, -0.0843,  0.5478,  0.3589,  0.5691,  0.2785, -0.4578,
           0.6471],
         [-0.5526, -0.0981,  0.5321,  0.3428,  0.5543,  0.2520, -0.4

In [7]:
class MultiHeadAttnImproved(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dropout=0, qkv_bias=False):
        super().__init__()

        assert(d_out % num_heads == 0)

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj= nn.Linear(d_out, d_out)

        self.dropout = nn.Dropout(dropout)
        

        self.register_buffer("mask",torch.triu(torch.ones(context_length, context_length), diagonal=1))



    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys    = self.W_query(x)
        values  = self.W_query(x)

        # split d_out into num_heads and head_dim
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # group by num_heads instead of num_tokens
        # b, num_tokens, num_heads, head_dim >> b, num_heads, num_tokens, head_dim
        queries = queries.transpose(1,2)
        keys    = keys.transpose(1,2)
        values  = values.transpose(1,2)

        # keys >> b, num_heads, head_dim, num_tokens
        # b, num_heads, num_tokens, num_tokens
        attn_scores = queries @ keys.transpose(2,3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_wts    = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim = -1)
        attn_wts    = self.dropout(attn_wts)

        # b, num_heads, num_tokens, head_dim >> b, num_tokens, num_heads, head_dim
        context_vec = (attn_wts @ values).transpose(1,2)

        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec



In [8]:
torch.manual_seed(123)
m_attn = MultiHeadAttnImproved(3, 4, 6, 2 )
c_vec = m_attn(batch)
c_vec

tensor([[[ 0.4217,  0.2751,  0.0218, -0.5732],
         [ 0.4949,  0.2928,  0.0227, -0.6655],
         [ 0.5160,  0.3004,  0.0227, -0.6933],
         [ 0.4776,  0.2921,  0.0118, -0.6802],
         [ 0.4559,  0.2911,  0.0088, -0.6630],
         [ 0.4494,  0.2863,  0.0043, -0.6685]],

        [[ 0.4217,  0.2751,  0.0218, -0.5732],
         [ 0.4949,  0.2928,  0.0227, -0.6655],
         [ 0.5160,  0.3004,  0.0227, -0.6933],
         [ 0.4776,  0.2921,  0.0118, -0.6802],
         [ 0.4559,  0.2911,  0.0088, -0.6630],
         [ 0.4494,  0.2863,  0.0043, -0.6685]]], grad_fn=<ViewBackward0>)

In [9]:
c_vec.shape

torch.Size([2, 6, 4])